In [16]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [17]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [18]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [19]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [20]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


In [21]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']



In [35]:
# 3) Initialize accumulators
fuel_demand_by_sector = pd.DataFrame({'time_period': base_case['time_period']}, index=base_case.index)

In [36]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [37]:
# 5) Loop over fuels and sectors
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    fuel_efficiency = base_case[eff_cols[0]]
    # find the demand column(s) for this fuel
    for sector in sectors:
        sector_dem_cols = [c for c in base_case.columns
                if (f'energy_demand_inen_{sector}' in c)]
        
        sector_fuel_fraction_cols = [c for c in base_case.columns
                if (f'frac_inen_energy_{sector}_{fuel}' in c)]

        if len(sector_fuel_fraction_cols)>0 and len(sector_dem_cols)>0:
            sector_total_demand = base_case[sector_dem_cols[0]]
            sector_fuel_fraction = base_case[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction*sector_total_demand).sum()>0 or fuel=='electricity':
                sector_fuel_demand = sector_fuel_fraction*sector_total_demand
                fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand
                if fuel=='electricity':
                    fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_electricity
                    fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_electricity
                else:
                    fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_other
                    fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_other
                sector_fuel_consumed = sector_fuel_demand/fuel_efficiency
                sector_fuel_consumed_baseline = sector_fuel_demand/fuel_efficiency.iloc[0]
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline-sector_fuel_consumed
                fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
                fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*opex_multiplier_efficiency


    


['energy_demand_inen_agriculture_and_livestock']
['energy_demand_inen_cement']
['energy_demand_inen_chemicals']
['energy_demand_inen_electronics']
['energy_demand_inen_glass']
['energy_demand_inen_lime_and_carbonite']
['energy_demand_inen_metals']
['energy_demand_inen_mining']
['energy_demand_inen_other_product_manufacturing']
['energy_demand_inen_paper']
['energy_demand_inen_plastic']
['energy_demand_inen_recycled_glass']
['energy_demand_inen_recycled_metals']
['energy_demand_inen_recycled_paper']
['energy_demand_inen_recycled_plastic']
['energy_demand_inen_recycled_rubber_and_leather']
['energy_demand_inen_recycled_textiles']
['energy_demand_inen_recycled_wood']
['energy_demand_inen_rubber_and_leather']
['energy_demand_inen_textiles']
[]
['energy_demand_inen_agriculture_and_livestock']
['energy_demand_inen_cement']
['energy_demand_inen_chemicals']
['energy_demand_inen_electronics']
['energy_demand_inen_glass']
['energy_demand_inen_lime_and_carbonite']
['energy_demand_inen_metals']
['

C:\Users\pkane\AppData\Local\Temp\ipykernel_5312\1487877770.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
C:\Users\pkane\AppData\Local\Temp\ipykernel_5312\1487877770.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
C:\Users\pkane\AppData\Local\Temp\ipykernel_5312\1487877770.py:33: PerformanceWarning: D

In [44]:
fuel_demand_by_sector['efficiency_energy_saving_plastic_natural_gas']

0     0.000000
1     0.000000
2     0.000000
3     0.000000
4     0.000000
5     0.000000
6     0.000000
7     0.094296
8     0.188855
9     0.284209
10    0.380809
11    0.479168
12    0.579770
13    0.683001
14    0.789144
15    0.898434
16    1.011110
17    1.127465
18    1.247851
19    1.372680
20    1.502406
21    1.637531
22    1.778596
23    1.926184
24    2.080913
25    2.243448
26    2.410148
27    2.585215
28    2.769497
29    2.963772
30    3.168855
31    3.385612
32    3.615023
33    3.858091
34    4.115920
35    4.389651
Name: efficiency_energy_saving_plastic_natural_gas, dtype: float64

In [26]:


# 6) Write out to CSV

fuel_demand_by_sector.to_csv(OUTPUT_DIR/'industrial_energy_cost.csv', index=False)
